In [ ]:
%pip install scikit-learn numpy pandas matplotlib

# Scikit-learn - Machine Learning End-to-End
Classification, regression, clustering, pipelines, cross-validation, and model evaluation.

In [ ]:
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

print('scikit-learn version:', sklearn.__version__)

## 1. Datasets

In [ ]:
from sklearn.datasets import load_iris, load_diabetes, make_classification

iris = load_iris(as_frame=True)
X_iris, y_iris = iris.data, iris.target
print('Iris shape:', X_iris.shape)
print('Classes:', iris.target_names)
X_iris.head()

## 2. Train / Test Split and Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train size: {X_train_s.shape[0]} | Test size: {X_test_s.shape[0]}')

## 3. Classification - Multiple Models

In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.svm             import SVC
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.metrics         import accuracy_score

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'SVM (RBF)':           SVC(kernel='rbf', C=1.0),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=0),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=0),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
}

rows = []
for name, clf in classifiers.items():
    clf.fit(X_train_s, y_train)
    train_acc = accuracy_score(y_train, clf.predict(X_train_s))
    test_acc  = accuracy_score(y_test,  clf.predict(X_test_s))
    rows.append({'Model': name, 'Train Acc': round(train_acc, 4), 'Test Acc': round(test_acc, 4)})

pd.DataFrame(rows).sort_values('Test Acc', ascending=False).reset_index(drop=True)

## 4. Confusion Matrix and Classification Report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

best_clf = RandomForestClassifier(n_estimators=100, random_state=0)
best_clf.fit(X_train_s, y_train)
y_pred = best_clf.predict(X_test_s)

print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    best_clf, X_test_s, y_test,
    display_labels=iris.target_names, ax=ax
)
ax.set_title('Random Forest - Confusion Matrix')
plt.tight_layout()
plt.show()

## 5. Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_train_s, y_train, cv=cv, scoring='accuracy')
    print(f'{name:<25}  CV Mean: {scores.mean():.4f}  Std: {scores.std():.4f}')

## 6. Regression

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

diabetes = load_diabetes(as_frame=True)
Xd, yd = diabetes.data, diabetes.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.2, random_state=0)

reg_models = {
    'Ridge':          Ridge(alpha=1.0),
    'Lasso':          Lasso(alpha=0.1),
    'Random Forest':  RandomForestRegressor(n_estimators=100, random_state=0),
}

rows_r = []
for name, reg in reg_models.items():
    reg.fit(Xd_tr, yd_tr)
    pred = reg.predict(Xd_te)
    rows_r.append({
        'Model': name,
        'RMSE': round(mean_squared_error(yd_te, pred) ** 0.5, 2),
        'R2':   round(r2_score(yd_te, pred), 4)
    })

pd.DataFrame(rows_r)

## 7. Pipeline

In [ ]:
from sklearn.pipeline      import Pipeline
from sklearn.decomposition import PCA

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=2)),
    ('clf',    LogisticRegression(max_iter=200))
])

pipe.fit(X_train, y_train)
pipe_acc = accuracy_score(y_test, pipe.predict(X_test))
print('Pipeline accuracy (StandardScaler -> PCA(2) -> LogReg):', pipe_acc)

## 8. Hyperparameter Tuning - GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'clf__C':       [0.01, 0.1, 1, 10],
    'clf__penalty': ['l2'],
    'clf__solver':  ['lbfgs']
}

grid_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=500))
])

gs = GridSearchCV(grid_pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
gs.fit(X_train, y_train)

print('Best params:', gs.best_params_)
print('Best CV score:', round(gs.best_score_, 4))
print('Test score:', round(accuracy_score(y_test, gs.predict(X_test)), 4))

## 9. Clustering - KMeans

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_scaled = StandardScaler().fit_transform(X_iris)

inertias   = []
silhouettes = []
ks = range(2, 9)

for k in ks:
    km = KMeans(n_clusters=k, random_state=0, n_init='auto')
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ks, inertias, 'o-', color='steelblue')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[1].plot(ks, silhouettes, 'o-', color='coral')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('k')
plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=0)
rf.fit(X_iris, y_iris)

importances = pd.Series(rf.feature_importances_, index=X_iris.columns)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 3))
importances_sorted.plot(kind='barh', ax=ax, color='mediumseagreen')
ax.set_title('Random Forest Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()